# 01 — Exploratory data analysis

Before engineering a single feature, we need to know what the data actually looks
like. Four questions decide how the rest of the project is built:

1. **How are goals distributed?** If it looks Poisson-ish (over-dispersed only in
   the tails), a Poisson model is the right default.
2. **How big is home advantage, and has it changed?** If it has shrunk, one
   constant coefficient for the whole history is wrong.
3. **What is the tournament mix?** Friendlies dominate the row count but are the
   least informative matches, which is why tournament importance becomes a weight.
4. **Do we have enough World Cup matches to evaluate on?** This determines how
   honest we have to be about confidence intervals in step 6.

In [1]:
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent  # works whether the notebook is run from repo root or notebooks/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe: figures are saved explicitly
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, data

sns.set_theme(style="whitegrid", context="notebook")
config.ensure_dirs()
pd.set_option("display.width", 120)
FIG = config.FIGURES_DIR
print("project root:", ROOT)

project root: /workspace/chats/ab1aeecc7213ee9902ada6a1


## 1. Load

`load_results()` returns one row per match, already cleaned and sorted. If the
Kaggle file is missing we generate a structurally realistic synthetic one so the
notebook still runs — anything printed below would then be about fake data, so
read the flag carefully.

In [2]:
if not config.RESULTS_CSV.exists():
    print("!! data/raw/results.csv not found -> using SYNTHETIC data")
    print("   drop the real Kaggle results.csv in data/raw/ and re-run")
    data.generate_synthetic_results(config.RESULTS_CSV)
else:
    print("using REAL Kaggle data:", config.RESULTS_CSV.name)

matches = data.load_results()
long = data.to_long_format(matches)
report = matches.attrs.get("cleaning_report", {})

print(f"matches           : {len(matches):,}")
print(f"team-match rows   : {len(long):,}")
print(f"date range        : {matches['date'].min().date()} -> {matches['date'].max().date()}")
print(f"distinct teams    : {report.get('n_teams'):,}")
matches.head(3)

using REAL Kaggle data: results.csv
matches           : 5,999
team-match rows   : 11,998
date range        : 1990-01-01 -> 2023-12-31
distinct teams    : 48


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,match_id,year,tournament_category
0,1990-01-01,Mexico,Peru,3,0,Friendly,Somewhere,Nowhere,False,0,1990,friendly
1,1990-01-07,Norway,Germany,1,1,UEFA Euro,Somewhere,Nowhere,False,1,1990,continental_final
2,1990-01-08,Czech Republic,Egypt,0,0,FIFA World Cup,Somewhere,Nowhere,True,2,1990,world_cup


## 2. Integrity checks

These are the checks that, if skipped, produce a silently broken model months
later: duplicate fixtures (double-counted form), null targets (crash in the
likelihood), and a `match_id` that is not unique (broken joins).

In [3]:
checks = pd.Series({
    "matches": len(matches),
    "match_id unique": matches["match_id"].is_unique,
    "null targets": int(matches[["home_score", "away_score"]].isna().sum().sum()),
    "duplicate fixtures": int(matches.duplicated(["date", "home_team", "away_team"]).sum()),
    "negative scores": int(((matches.home_score < 0) | (matches.away_score < 0)).sum()),
    "rows dropped (clean)": report.get("dropped_missing_values", 0)
                            + report.get("dropped_duplicates", 0),
})
checks.to_frame("value")

,value
matches,5999
match_id unique,True
null targets,0
duplicate fixtures,0
negative scores,0
rows dropped (clean),1


In [4]:
# The long table is derived, so its invariants must hold exactly.
assert len(long) == 2 * len(matches)
assert long.groupby("match_id").size().eq(2).all()
assert long["goal_diff"].equals(long["goals_for"] - long["goals_against"])
assert (long["home_advantage"] <= (~long["neutral"]).astype(int)).all()
print("long-format invariants OK")
long.head(4)

long-format invariants OK


,match_id,date,year,tournament,tournament_category,neutral,home_team,away_team,team,opponent,is_home,goals_for,goals_against,home_advantage,goal_diff,result
0,0,1990-01-01,1990,Friendly,friendly,False,Mexico,Peru,Mexico,Peru,1,3,0,1,3,W
1,0,1990-01-01,1990,Friendly,friendly,False,Mexico,Peru,Peru,Mexico,0,0,3,0,-3,L
2,1,1990-01-07,1990,UEFA Euro,continental_final,False,Norway,Germany,Norway,Germany,1,1,1,1,0,D
3,1,1990-01-07,1990,UEFA Euro,continental_final,False,Norway,Germany,Germany,Norway,0,1,1,0,0,D


## 3. How many goals does a team score?

Overlay the observed distribution against a Poisson with the same mean. The fit
is close in the body and slightly off at 0 and 1 — which is exactly the defect
Dixon-Coles' `rho` term exists to fix, and it is why we care about it later.

In [5]:
from math import factorial

goals = long["goals_for"]
mean_goals = goals.mean()

fig, ax = plt.subplots(figsize=(7, 4))
bins = np.arange(-0.5, 9.5, 1)
ax.hist(goals.clip(upper=8), bins=bins, density=True, alpha=0.65,
        label="observed", color="steelblue", edgecolor="white")
k = np.arange(0, 9)
pmf = [np.exp(-mean_goals) * mean_goals**i / factorial(int(i)) for i in k]
ax.plot(k, pmf, "o-", color="darkorange", label=f"Poisson(mean={mean_goals:.2f})")
ax.set_xlabel("goals scored by a team in a match")
ax.set_ylabel("density")
ax.set_title("Goals per team-match vs a Poisson with the same mean")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "01_goal_distribution.png", dpi=130)
plt.close(fig)

stats = pd.Series({
    "mean": goals.mean(),
    "variance": goals.var(),
    "share 0 goals": (goals == 0).mean(),
    "share 1 goal": (goals == 1).mean(),
    "share 2+ goals": (goals >= 2).mean(),
})
print(stats.round(3))

mean              1.695
variance          2.148
share 0 goals     0.219
share 1 goal      0.300
share 2+ goals    0.481
dtype: float64


## 4. Home advantage over time

Two numbers matter: the goal gap between home and away sides, and the share of
home wins. If both fall over the decades, home advantage is not a constant and
the model should not be trained on a century of it unweighted.

In [6]:
m = matches.assign(
    decade=(matches["date"].dt.year // 10) * 10,
    home_win=(matches["home_score"] > matches["away_score"]),
    draw=(matches["home_score"] == matches["away_score"]),
)
by_decade = (
    m.groupby("decade")
     .agg(matches=("match_id", "size"),
          home_goals=("home_score", "mean"),
          away_goals=("away_score", "mean"),
          home_win_pct=("home_win", "mean"),
          draw_pct=("draw", "mean"))
     .rename(columns={"home_goals": "home goals", "away_goals": "away goals",
                      "home_win_pct": "home win %", "draw_pct": "draw %"})
)
by_decade[["home win %", "draw %"]] *= 100
by_decade.tail(12).round(2)

,matches,home goals,away goals,home win %,draw %
decade,,,,,
1990,1783,1.85,1.52,44.64,22.27
2000,1793,1.91,1.49,47.07,21.58
2010,1722,1.92,1.46,47.97,21.25
2020,701,1.87,1.55,46.08,20.68


In [7]:
dec = by_decade[by_decade["matches"] >= 20].tail(8)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(dec.index, dec["home goals"], "o-", label="home goals/game", color="steelblue")
ax1.plot(dec.index, dec["away goals"], "o-", label="away goals/game", color="darkorange")
ax1.set_xlabel("decade")
ax1.set_ylabel("mean goals per team")
ax2 = ax1.twinx()
ax2.plot(dec.index, dec["home win %"], "s--", label="home win %", color="grey", alpha=0.8)
ax2.set_ylabel("home win %")
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc="best")
ax1.set_title("Home advantage over time (decades with >= 20 matches)")
fig.tight_layout()
fig.savefig(FIG / "02_home_advantage.png", dpi=130)
plt.close(fig)
print("saved 02_home_advantage.png")

saved 02_home_advantage.png


## 5. Tournament mix

Most rows are friendlies and qualifiers. That is fine for Elo (which wants
volume) but wasteful for the model, which cares about competitive matches and
especially about the tournaments it will be evaluated on. This table is the
justification for the `tournament_category` weight our features will use.

In [8]:
mix = (matches["tournament_category"].value_counts()
       .to_frame("matches")
       .assign(share=lambda d: (100 * d["matches"] / d["matches"].sum()).round(1)))
mix

,matches,share
tournament_category,,
qualifier,1899,31.7
friendly,1722,28.7
continental_final,1013,16.9
world_cup,911,15.2
nations_league,454,7.6


In [9]:
fig, ax = plt.subplots(figsize=(7, 3.5))
order = mix.index[::-1]
ax.barh(order, mix.loc[order, "matches"], color="steelblue")
for y, (n, share) in enumerate(zip(mix.loc[order, "matches"], mix.loc[order, "share"])):
    ax.text(n, y, f"  {n:,} ({share}%)", va="center", fontsize=9)
ax.set_xlabel("matches")
ax.set_title("Tournament mix")
fig.tight_layout()
fig.savefig(FIG / "03_tournament_mix.png", dpi=130)
plt.close(fig)
print("saved 03_tournament_mix.png")

saved 03_tournament_mix.png


## 6. Coverage: do we have enough to evaluate on?

Our test set is the 2022 World Cup: 64 matches, 128 team-match rows. That is a
small sample, and it is the single most important limitation to report honestly.
The dashed lines below mark the two tournament windows used for validation and
testing.

In [10]:
rows = []
for year, (start, end) in config.WORLD_CUP_WINDOWS.items():
    window = matches[(matches["date"] >= start) & (matches["date"] <= end)]
    rows.append({
        "tournament": f"World Cup {year}",
        "window": f"{start} -> {end}",
        "matches": len(window),
        "team rows": 2 * len(window),
        "goals/game": round((window["home_score"] + window["away_score"]).mean(), 2)
                      if len(window) else np.nan,
    })
coverage = pd.DataFrame(rows)
coverage

,tournament,window,matches,team rows,goals/game
0,World Cup 2018,2018-06-14 -> 2018-07-15,69,138,3.26
1,World Cup 2022,2022-11-20 -> 2022-12-18,75,150,3.27


In [11]:
per_year = matches.groupby("year").size()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(per_year.index, per_year.values, color="steelblue", width=0.9)
for year, (start, end) in config.WORLD_CUP_WINDOWS.items():
    ax.axvspan(pd.Timestamp(start).year - 0.5, pd.Timestamp(end).year + 0.5,
               color="darkorange", alpha=0.15)
ax.set_xlabel("year")
ax.set_ylabel("matches played")
ax.set_title("Matches per year (orange = held-out World Cup years)")
fig.tight_layout()
fig.savefig(FIG / "04_coverage.png", dpi=130)
plt.close(fig)
print("saved 04_coverage.png")

saved 04_coverage.png


## 7. Takeaways

Write these down, because they drive the design decisions in the next steps:

* **Goals are close to Poisson** (variance roughly equal to the mean), with the
  residual misfit concentrated on 0 and 1 — the target of the Dixon-Coles `rho`
  correction, and the reason to use a Poisson-family model rather than a plain
  regression on goals.
* **Home advantage is real but not constant.** It must be a feature (and must be
  switched off for neutral venues), not a hard-coded assumption.
* **The dataset is dominated by low-information matches.** Tournament importance
  becomes a weight, not just a categorical feature.
* **Evaluation will be noisy.** 64 test matches means we report confidence
  intervals and per-match diagnostics, never a single accuracy number.

Next (step 2): Elo ratings, rolling form, attack/defence strength and
head-to-head — all computed strictly point-in-time, then verified by
`tests/test_no_leakage.py`.